In [1]:
!pip -q install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

import platform
import os
import time
import psutil

from datetime import datetime
from zoneinfo import ZoneInfo

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("INFOREST_BigData")
    .getOrCreate()
)

print("=== ENTORNO BIG DATA ===")
print("Spark:", spark.version)
print("Python:", platform.python_version())
print("CPU lógicos disponibles:", os.cpu_count())
print("RAM disponible aproximada (GB):",
      round(psutil.virtual_memory().total / (1024**3), 2))
print("Paralelismo Spark:", spark.sparkContext.defaultParallelism)
print("Fecha de ejecución:",
      datetime.now(ZoneInfo("America/Lima")).strftime("%Y-%m-%d %H:%M:%S"))

=== ENTORNO BIG DATA ===
Spark: 4.0.4
Python: 3.13.16
CPU lógicos disponibles: 2
RAM disponible aproximada (GB): 12.67
Paralelismo Spark: 2
Fecha de ejecución: 2026-10-10 22:43:05


In [3]:
!java -version

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)


In [10]:
from google.colab import files

uploaded = files.upload()

Saving DW_DIM_CAJA.csv to DW_DIM_CAJA (1).csv
Saving DW_DIM_CLIENTE.csv to DW_DIM_CLIENTE (1).csv
Saving DW_DIM_FECHA.csv to DW_DIM_FECHA (1).csv
Saving DW_DIM_FORMA_PAGO.csv to DW_DIM_FORMA_PAGO (1).csv
Saving DW_DIM_MOZO.csv to DW_DIM_MOZO (1).csv
Saving DW_DIM_PRODUCTO.csv to DW_DIM_PRODUCTO (1).csv
Saving DW_FACT_PAGOS.csv to DW_FACT_PAGOS (1).csv
Saving DW_FACT_VENTAS.csv to DW_FACT_VENTAS (1).csv


In [11]:
print("=== ARCHIVOS SUBIDOS ===")

for nombre in uploaded.keys():
    ruta = "/content/" + nombre
    tamano_kb = os.path.getsize(ruta) / 1024

    print(f"{nombre}: {tamano_kb:.2f} KB")

=== ARCHIVOS SUBIDOS ===
DW_DIM_CAJA (1).csv: 0.04 KB
DW_DIM_CLIENTE (1).csv: 1.10 KB
DW_DIM_FECHA (1).csv: 6.08 KB
DW_DIM_FORMA_PAGO (1).csv: 0.08 KB
DW_DIM_MOZO (1).csv: 0.09 KB
DW_DIM_PRODUCTO (1).csv: 1.32 KB
DW_FACT_PAGOS (1).csv: 263.12 KB
DW_FACT_VENTAS (1).csv: 983.59 KB


In [12]:
archivos = [
    "DW_FACT_VENTAS.csv",
    "DW_FACT_PAGOS.csv",
    "DW_DIM_FECHA.csv",
    "DW_DIM_PRODUCTO.csv",
    "DW_DIM_CLIENTE.csv",
    "DW_DIM_MOZO.csv",
    "DW_DIM_CAJA.csv",
    "DW_DIM_FORMA_PAGO.csv"
]

print("=== PRIMERA LÍNEA DE CADA CSV ===")

for archivo in archivos:

    print("\n" + archivo)

    with open(
        "/content/" + archivo,
        "r",
        encoding="utf-8-sig"
    ) as f:

        print(f.readline().strip())

=== PRIMERA LÍNEA DE CADA CSV ===

DW_FACT_VENTAS.csv
0;155;5;90;3;0;PED0000001;B001-00000001;1;22;0;3.36;0;22

DW_FACT_PAGOS.csv
0;155;1;0;90;B001-00000001;001;39;0

DW_DIM_FECHA.csv
0;2026-05-17;17;5;Mayo;2;2026

DW_DIM_PRODUCTO.csv
0;P000001;01;0101;Pollo a la brasa 1/4;1/4 Pollo;HOR;18;1

DW_DIM_CLIENTE.csv
idCliente;tCodigoCliente

DW_DIM_MOZO.csv
0;M006

DW_DIM_CAJA.csv
0;C1

DW_DIM_FORMA_PAGO.csv
0;01;NULL


In [13]:
def cargar_sin_encabezado(nombre, columnas):
    df = (
        spark.read
        .option("header", "false")
        .option("sep", ";")
        .option("inferSchema", "true")
        .option("nullValue", "NULL")
        .csv("/content/" + nombre)
    )

    return df.toDF(*columnas)


def cargar_con_encabezado(nombre):
    return (
        spark.read
        .option("header", "true")
        .option("sep", ";")
        .option("inferSchema", "true")
        .option("nullValue", "NULL")
        .csv("/content/" + nombre)
    )

In [14]:
fact_ventas = cargar_sin_encabezado(
    "DW_FACT_VENTAS.csv",
    [
        "idVenta",
        "idFecha",
        "idProducto",
        "idCliente",
        "idMozo",
        "idCaja",
        "tCodigoPedido",
        "tDocumento",
        "cantidad",
        "precioUnitario",
        "descuento",
        "impuesto1",
        "impuesto2",
        "venta"
    ]
)

In [15]:
fact_pagos = cargar_sin_encabezado(
    "DW_FACT_PAGOS.csv",
    [
        "idPago",
        "idFecha",
        "idFormaPago",
        "idCaja",
        "idCliente",
        "tDocumento",
        "tCorrelativo",
        "monto",
        "propina"
    ]
)

In [16]:
dim_fecha = cargar_sin_encabezado(
    "DW_DIM_FECHA.csv",
    [
        "idFecha",
        "fecha",
        "dia",
        "mes",
        "nombreMes",
        "trimestre",
        "anio"
    ]
)

In [17]:
dim_producto = cargar_sin_encabezado(
    "DW_DIM_PRODUCTO.csv",
    [
        "idProducto",
        "tCodigoProducto",
        "tGrupo",
        "tSubGrupo",
        "tDetallado",
        "tResumido",
        "tArea",
        "nPrecioVenta",
        "lActivo"
    ]
)

In [18]:
dim_cliente = cargar_con_encabezado(
    "DW_DIM_CLIENTE.csv"
)

In [19]:
dim_mozo = cargar_sin_encabezado(
    "DW_DIM_MOZO.csv",
    [
        "idMozo",
        "tMozo"
    ]
)

dim_caja = cargar_sin_encabezado(
    "DW_DIM_CAJA.csv",
    [
        "idCaja",
        "tCaja"
    ]
)

dim_forma_pago = cargar_sin_encabezado(
    "DW_DIM_FORMA_PAGO.csv",
    [
        "idFormaPago",
        "tTipoPago",
        "tTarjeta"
    ]
)

In [20]:
datasets = {
    "DW_FACT_VENTAS": fact_ventas,
    "DW_FACT_PAGOS": fact_pagos,
    "DW_DIM_FECHA": dim_fecha,
    "DW_DIM_PRODUCTO": dim_producto,
    "DW_DIM_CLIENTE": dim_cliente,
    "DW_DIM_MOZO": dim_mozo,
    "DW_DIM_CAJA": dim_caja,
    "DW_DIM_FORMA_PAGO": dim_forma_pago
}

print("=== CONTEO DE REGISTROS EN SPARK ===")

for nombre, df in datasets.items():
    print(nombre, "=", df.count())

=== CONTEO DE REGISTROS EN SPARK ===
DW_FACT_VENTAS = 15000
DW_FACT_PAGOS = 6000
DW_DIM_FECHA = 180
DW_DIM_PRODUCTO = 20
DW_DIM_CLIENTE = 100
DW_DIM_MOZO = 8
DW_DIM_CAJA = 3
DW_DIM_FORMA_PAGO = 4


In [21]:
print("=== COLUMNAS DETECTADAS ===")

for nombre, df in datasets.items():
    print(nombre, "->", df.columns)

=== COLUMNAS DETECTADAS ===
DW_FACT_VENTAS -> ['idVenta', 'idFecha', 'idProducto', 'idCliente', 'idMozo', 'idCaja', 'tCodigoPedido', 'tDocumento', 'cantidad', 'precioUnitario', 'descuento', 'impuesto1', 'impuesto2', 'venta']
DW_FACT_PAGOS -> ['idPago', 'idFecha', 'idFormaPago', 'idCaja', 'idCliente', 'tDocumento', 'tCorrelativo', 'monto', 'propina']
DW_DIM_FECHA -> ['idFecha', 'fecha', 'dia', 'mes', 'nombreMes', 'trimestre', 'anio']
DW_DIM_PRODUCTO -> ['idProducto', 'tCodigoProducto', 'tGrupo', 'tSubGrupo', 'tDetallado', 'tResumido', 'tArea', 'nPrecioVenta', 'lActivo']
DW_DIM_CLIENTE -> ['idCliente', 'tCodigoCliente']
DW_DIM_MOZO -> ['idMozo', 'tMozo']
DW_DIM_CAJA -> ['idCaja', 'tCaja']
DW_DIM_FORMA_PAGO -> ['idFormaPago', 'tTipoPago', 'tTarjeta']


In [22]:
fact_ventas.show(3, truncate=False)
fact_pagos.show(3, truncate=False)
dim_producto.show(3, truncate=False)
dim_forma_pago.show(10, truncate=False)

+-------+-------+----------+---------+------+------+-------------+-------------+--------+--------------+---------+---------+---------+-----+
|idVenta|idFecha|idProducto|idCliente|idMozo|idCaja|tCodigoPedido|tDocumento   |cantidad|precioUnitario|descuento|impuesto1|impuesto2|venta|
+-------+-------+----------+---------+------+------+-------------+-------------+--------+--------------+---------+---------+---------+-----+
|0      |155    |5         |90       |3     |0     |PED0000001   |B001-00000001|1       |22            |0        |3.36     |0        |22   |
|1      |155    |10        |90       |3     |0     |PED0000001   |B001-00000001|1       |5             |0        |0.76     |0        |5    |
|2      |155    |15        |90       |3     |0     |PED0000001   |B001-00000001|1       |12            |0        |1.83     |0        |12   |
+-------+-------+----------+---------+------+------+-------------+-------------+--------+--------------+---------+---------+---------+-----+
only showing 

In [23]:
# ==========================================
# NORMALIZACIÓN DE CÓDIGOS
# ==========================================

dim_producto = (
    dim_producto
    .withColumn(
        "tGrupo",
        F.lpad(F.col("tGrupo").cast("string"), 2, "0")
    )
    .withColumn(
        "tSubGrupo",
        F.lpad(F.col("tSubGrupo").cast("string"), 4, "0")
    )
)

dim_forma_pago = (
    dim_forma_pago
    .withColumn(
        "tTipoPago",
        F.lpad(F.col("tTipoPago").cast("string"), 2, "0")
    )
)

fact_pagos = (
    fact_pagos
    .withColumn(
        "tCorrelativo",
        F.lpad(F.col("tCorrelativo").cast("string"), 3, "0")
    )
)

print("=== CÓDIGOS NORMALIZADOS ===")

dim_producto.select(
    "tCodigoProducto",
    "tGrupo",
    "tSubGrupo"
).show(5, truncate=False)

dim_forma_pago.show(truncate=False)

fact_pagos.select(
    "tDocumento",
    "tCorrelativo"
).show(5, truncate=False)

=== CÓDIGOS NORMALIZADOS ===
+---------------+------+---------+
|tCodigoProducto|tGrupo|tSubGrupo|
+---------------+------+---------+
|P000001        |01    |0101     |
|P000002        |01    |0101     |
|P000003        |01    |0101     |
|P000004        |01    |0102     |
|P000005        |01    |0102     |
+---------------+------+---------+
only showing top 5 rows
+-----------+---------+--------+
|idFormaPago|tTipoPago|tTarjeta|
+-----------+---------+--------+
|0          |01       |NULL    |
|1          |02       |VI      |
|2          |02       |MC      |
|3          |03       |NULL    |
+-----------+---------+--------+

+-------------+------------+
|tDocumento   |tCorrelativo|
+-------------+------------+
|B001-00000001|001         |
|B001-00000002|001         |
|B001-00000003|001         |
|B001-00000005|001         |
|B001-00000005|002         |
+-------------+------------+
only showing top 5 rows


In [24]:
datasets = {
    "DW_FACT_VENTAS": fact_ventas,
    "DW_FACT_PAGOS": fact_pagos,
    "DW_DIM_FECHA": dim_fecha,
    "DW_DIM_PRODUCTO": dim_producto,
    "DW_DIM_CLIENTE": dim_cliente,
    "DW_DIM_MOZO": dim_mozo,
    "DW_DIM_CAJA": dim_caja,
    "DW_DIM_FORMA_PAGO": dim_forma_pago
}

In [25]:
print("=== CALIDAD DW_FACT_VENTAS ===")

fact_ventas.select(
    F.sum(F.col("idFecha").isNull().cast("int")).alias("SinFecha"),
    F.sum(F.col("idProducto").isNull().cast("int")).alias("SinProducto"),
    F.sum(F.col("idCliente").isNull().cast("int")).alias("SinCliente"),
    F.sum(F.col("idMozo").isNull().cast("int")).alias("SinMozo"),
    F.sum(F.col("idCaja").isNull().cast("int")).alias("SinCaja"),
    F.sum(F.col("cantidad").isNull().cast("int")).alias("SinCantidad"),
    F.sum(F.col("venta").isNull().cast("int")).alias("SinVenta")
).show()

=== CALIDAD DW_FACT_VENTAS ===
+--------+-----------+----------+-------+-------+-----------+--------+
|SinFecha|SinProducto|SinCliente|SinMozo|SinCaja|SinCantidad|SinVenta|
+--------+-----------+----------+-------+-------+-----------+--------+
|       0|          0|         0|      0|      0|          0|       0|
+--------+-----------+----------+-------+-------+-----------+--------+



In [26]:
print("=== CALIDAD DW_FACT_PAGOS ===")

fact_pagos.select(
    F.sum(F.col("idFecha").isNull().cast("int")).alias("SinFecha"),
    F.sum(F.col("idFormaPago").isNull().cast("int")).alias("SinFormaPago"),
    F.sum(F.col("idCliente").isNull().cast("int")).alias("SinCliente"),
    F.sum(F.col("idCaja").isNull().cast("int")).alias("SinCaja"),
    F.sum(F.col("monto").isNull().cast("int")).alias("SinMonto")
).show()

=== CALIDAD DW_FACT_PAGOS ===
+--------+------------+----------+-------+--------+
|SinFecha|SinFormaPago|SinCliente|SinCaja|SinMonto|
+--------+------------+----------+-------+--------+
|       0|           0|         0|      0|       0|
+--------+------------+----------+-------+--------+



In [27]:
print("=== KPI VENTAS ===")

fact_ventas.agg(
    F.count("*").alias("RegistrosVentas"),
    F.countDistinct("tDocumento").alias("Documentos"),
    F.sum("cantidad").alias("UnidadesVendidas"),
    F.sum("venta").alias("VentasTotales"),
    (
        F.sum("venta") /
        F.countDistinct("tDocumento")
    ).alias("TicketPromedio")
).show()

=== KPI VENTAS ===
+---------------+----------+----------------+-------------+--------------+
|RegistrosVentas|Documentos|UnidadesVendidas|VentasTotales|TicketPromedio|
+---------------+----------+----------------+-------------+--------------+
|          15000|      5000|           17142|       365122|       73.0244|
+---------------+----------+----------------+-------------+--------------+



In [28]:
print("=== KPI PAGOS ===")

fact_pagos.agg(
    F.count("*").alias("RegistrosPagos"),
    F.sum("monto").alias("TotalPagos"),
    F.sum("propina").alias("TotalPropinas")
).show()

=== KPI PAGOS ===
+--------------+----------+-------------+
|RegistrosPagos|TotalPagos|TotalPropinas|
+--------------+----------+-------------+
|          6000|  365122.0|         1000|
+--------------+----------+-------------+



In [29]:
venta_total = fact_ventas.agg(
    F.sum("venta")
).first()[0]

pago_total = fact_pagos.agg(
    F.sum("monto")
).first()[0]

diferencia = venta_total - pago_total

print("=== RECONCILIACIÓN ===")
print("Venta total:", venta_total)
print("Pago total:", pago_total)
print("Diferencia:", diferencia)

if abs(diferencia) < 0.01:
    print("RESULTADO: RECONCILIACIÓN CORRECTA")
else:
    print("RESULTADO: EXISTE DIFERENCIA")

=== RECONCILIACIÓN ===
Venta total: 365122
Pago total: 365122.0
Diferencia: 0.0
RESULTADO: RECONCILIACIÓN CORRECTA


In [30]:
top_productos = (
    fact_ventas.alias("V")
    .join(
        dim_producto.alias("P"),
        F.col("V.idProducto") == F.col("P.idProducto"),
        "inner"
    )
    .groupBy(
        F.col("P.tCodigoProducto"),
        F.col("P.tDetallado")
    )
    .agg(
        F.sum("V.cantidad").alias("CantidadVendida"),
        F.sum("V.venta").alias("VentaTotal")
    )
    .orderBy(F.desc("VentaTotal"))
)

print("=== TOP 5 PRODUCTOS ===")
top_productos.show(5, truncate=False)

=== TOP 5 PRODUCTOS ===
+---------------+-----------------------+---------------+----------+
|tCodigoProducto|tDetallado             |CantidadVendida|VentaTotal|
+---------------+-----------------------+---------------+----------+
|P000020        |Combo familiar         |857            |64275     |
|P000003        |Pollo a la brasa entero|857            |49706     |
|P000004        |Parrilla personal      |856            |29960     |
|P000002        |Pollo a la brasa 1/2   |858            |27456     |
|P000005        |Lomo saltado           |857            |24853     |
+---------------+-----------------------+---------------+----------+
only showing top 5 rows


In [31]:
ventas_mozo = (
    fact_ventas.alias("V")
    .join(
        dim_mozo.alias("M"),
        F.col("V.idMozo") == F.col("M.idMozo"),
        "inner"
    )
    .groupBy(F.col("M.tMozo"))
    .agg(
        F.sum("V.cantidad").alias("CantidadVendida"),
        F.sum("V.venta").alias("VentaTotal")
    )
    .orderBy(F.desc("VentaTotal"))
)

print("=== VENTAS POR MOZO ===")
ventas_mozo.show(truncate=False)

=== VENTAS POR MOZO ===
+-----+---------------+----------+
|tMozo|CantidadVendida|VentaTotal|
+-----+---------------+----------+
|M005 |2144           |50326     |
|M001 |2142           |50261     |
|M006 |2143           |48436     |
|M002 |2142           |48331     |
|M003 |2143           |43599     |
|M007 |2142           |43554     |
|M004 |2144           |40316     |
|M008 |2142           |40299     |
+-----+---------------+----------+



In [32]:
ventas_caja = (
    fact_ventas.alias("V")
    .join(
        dim_caja.alias("C"),
        F.col("V.idCaja") == F.col("C.idCaja"),
        "inner"
    )
    .groupBy(F.col("C.tCaja"))
    .agg(
        F.sum("V.venta").alias("VentaTotal")
    )
    .orderBy(F.desc("VentaTotal"))
)

print("=== VENTAS POR CAJA ===")
ventas_caja.show(truncate=False)

=== VENTAS POR CAJA ===
+-----+----------+
|tCaja|VentaTotal|
+-----+----------+
|C1   |121756    |
|C2   |121704    |
|C3   |121662    |
+-----+----------+



In [33]:
dim_forma_pago_desc = (
    dim_forma_pago
    .withColumn(
        "FormaPago",
        F.when(F.col("tTipoPago") == "01", "Efectivo")
         .when(F.col("tTipoPago") == "02", "Tarjeta")
         .when(F.col("tTipoPago") == "03", "Cheque")
         .otherwise("Otro")
    )
    .withColumn(
        "TarjetaDescripcion",
        F.when(F.col("tTarjeta") == "VI", "Visa")
         .when(F.col("tTarjeta") == "MC", "Mastercard")
         .otherwise(F.lit(None))
    )
)

In [34]:
pagos_tipo = (
    fact_pagos.alias("P")
    .join(
        dim_forma_pago_desc.alias("FP"),
        F.col("P.idFormaPago") == F.col("FP.idFormaPago"),
        "inner"
    )
    .groupBy(
        F.col("FP.FormaPago"),
        F.col("FP.TarjetaDescripcion")
    )
    .agg(
        F.count("*").alias("CantidadOperaciones"),
        F.sum("P.monto").alias("MontoTotal"),
        F.sum("P.propina").alias("PropinaTotal")
    )
    .orderBy(F.desc("MontoTotal"))
)

print("=== PAGOS POR TIPO ===")
pagos_tipo.show(truncate=False)

=== PAGOS POR TIPO ===
+---------+------------------+-------------------+------------------+------------+
|FormaPago|TarjetaDescripcion|CantidadOperaciones|MontoTotal        |PropinaTotal|
+---------+------------------+-------------------+------------------+------------+
|Tarjeta  |Visa              |1667               |106432.79999999986|334         |
|Efectivo |NULL              |1666               |106374.39999999985|332         |
|Cheque   |NULL              |1667               |106373.99999999985|334         |
|Tarjeta  |Mastercard        |1000               |45940.79999999958 |0           |
+---------+------------------+-------------------+------------------+------------+



In [35]:
print("=== GENERACIÓN DE PARQUET ===")

ventas_producto_parquet = (
    fact_ventas.alias("V")
    .join(
        dim_producto.alias("P"),
        F.col("V.idProducto") == F.col("P.idProducto"),
        "inner"
    )
    .groupBy(
        F.col("P.tCodigoProducto"),
        F.col("P.tDetallado"),
        F.col("P.tGrupo"),
        F.col("P.tSubGrupo")
    )
    .agg(
        F.sum("V.cantidad").alias("CantidadVendida"),
        F.sum("V.venta").alias("VentaTotal"),
        F.sum("V.descuento").alias("DescuentoTotal")
    )
)

ruta_parquet = "/content/INFOREST_RESULTADO_PARQUET"

ventas_producto_parquet.write \
    .mode("overwrite") \
    .parquet(ruta_parquet)

resultado_parquet = spark.read.parquet(ruta_parquet)

print("Registros generados en Parquet:", resultado_parquet.count())

resultado_parquet \
    .orderBy(F.desc("VentaTotal")) \
    .show(20, truncate=False)

=== GENERACIÓN DE PARQUET ===
Registros generados en Parquet: 20
+---------------+-----------------------+------+---------+---------------+----------+--------------+
|tCodigoProducto|tDetallado             |tGrupo|tSubGrupo|CantidadVendida|VentaTotal|DescuentoTotal|
+---------------+-----------------------+------+---------+---------------+----------+--------------+
|P000020        |Combo familiar         |05    |0501     |857            |64275     |0             |
|P000003        |Pollo a la brasa entero|01    |0101     |857            |49706     |0             |
|P000004        |Parrilla personal      |01    |0102     |856            |29960     |0             |
|P000002        |Pollo a la brasa 1/2   |01    |0101     |858            |27456     |0             |
|P000005        |Lomo saltado           |01    |0102     |857            |24853     |0             |
|P000007        |Tallarín saltado       |01    |0103     |857            |20568     |0             |
|P000006        |Arroz cha

In [36]:
print("=== ESQUEMA DEL PARQUET ===")
resultado_parquet.printSchema()

=== ESQUEMA DEL PARQUET ===
root
 |-- tCodigoProducto: string (nullable = true)
 |-- tDetallado: string (nullable = true)
 |-- tGrupo: string (nullable = true)
 |-- tSubGrupo: string (nullable = true)
 |-- CantidadVendida: long (nullable = true)
 |-- VentaTotal: long (nullable = true)
 |-- DescuentoTotal: long (nullable = true)



In [37]:
print("=== VALIDACIÓN DEL PARQUET ===")

resultado_parquet.agg(
    F.sum("CantidadVendida").alias("CantidadVendida"),
    F.sum("VentaTotal").alias("VentaTotal")
).show()

=== VALIDACIÓN DEL PARQUET ===
+---------------+----------+
|CantidadVendida|VentaTotal|
+---------------+----------+
|          17142|    365122|
+---------------+----------+



In [38]:
print("=== CALENTAMIENTO DEL MOTOR SPARK ===")

fact_ventas.limit(100).collect()
dim_producto.collect()

print("Spark preparado para las pruebas.")

=== CALENTAMIENTO DEL MOTOR SPARK ===
Spark preparado para las pruebas.


In [39]:
def ejecutar_experimento(factor, repeticion):

    # Limpiar caché antes de cada ejecución
    spark.catalog.clearCache()

    # Crear factor de replicación
    replicas = (
        spark.range(factor)
        .withColumnRenamed("id", "replica")
    )

    # Escalar el conjunto de ventas
    ventas_escaladas = (
        fact_ventas
        .crossJoin(F.broadcast(replicas))
    )

    inicio = time.perf_counter()

    # Operación analítica representativa:
    # JOIN + GROUP BY + SUM
    resultado = (
        ventas_escaladas.alias("V")
        .join(
            F.broadcast(dim_producto.alias("P")),
            F.col("V.idProducto") == F.col("P.idProducto"),
            "inner"
        )
        .groupBy(
            F.col("P.tCodigoProducto"),
            F.col("P.tDetallado")
        )
        .agg(
            F.count("*").alias("Registros"),
            F.sum("V.cantidad").alias("CantidadVendida"),
            F.sum("V.venta").alias("VentaTotal")
        )
    )

    # collect() fuerza la ejecución real de Spark
    filas = resultado.collect()

    fin = time.perf_counter()

    tiempo = fin - inicio

    registros = sum(
        fila["Registros"]
        for fila in filas
    )

    unidades = sum(
        fila["CantidadVendida"] or 0
        for fila in filas
    )

    venta = sum(
        fila["VentaTotal"] or 0
        for fila in filas
    )

    throughput = (
        registros / tiempo
        if tiempo > 0
        else 0
    )

    return {
        "Factor": factor,
        "Repeticion": repeticion,
        "Registros": int(registros),
        "TiempoSegundos": round(tiempo, 4),
        "RegistrosPorSegundo": round(throughput, 2),
        "UnidadesVendidas": float(unidades),
        "VentaTotal": round(float(venta), 2)
    }

In [40]:
resultados = []

for factor in [1, 5, 10]:

    print(f"\n===== ESCENARIO {factor}x =====")

    for repeticion in range(1, 4):

        r = ejecutar_experimento(
            factor,
            repeticion
        )

        resultados.append(r)

        print(r)


===== ESCENARIO 1x =====
{'Factor': 1, 'Repeticion': 1, 'Registros': 15000, 'TiempoSegundos': 0.5551, 'RegistrosPorSegundo': 27024.35, 'UnidadesVendidas': 17142.0, 'VentaTotal': 365122.0}
{'Factor': 1, 'Repeticion': 2, 'Registros': 15000, 'TiempoSegundos': 0.6236, 'RegistrosPorSegundo': 24053.69, 'UnidadesVendidas': 17142.0, 'VentaTotal': 365122.0}
{'Factor': 1, 'Repeticion': 3, 'Registros': 15000, 'TiempoSegundos': 0.4102, 'RegistrosPorSegundo': 36564.75, 'UnidadesVendidas': 17142.0, 'VentaTotal': 365122.0}

===== ESCENARIO 5x =====
{'Factor': 5, 'Repeticion': 1, 'Registros': 75000, 'TiempoSegundos': 0.3853, 'RegistrosPorSegundo': 194642.56, 'UnidadesVendidas': 85710.0, 'VentaTotal': 1825610.0}
{'Factor': 5, 'Repeticion': 2, 'Registros': 75000, 'TiempoSegundos': 0.5577, 'RegistrosPorSegundo': 134486.85, 'UnidadesVendidas': 85710.0, 'VentaTotal': 1825610.0}
{'Factor': 5, 'Repeticion': 3, 'Registros': 75000, 'TiempoSegundos': 0.4692, 'RegistrosPorSegundo': 159833.12, 'UnidadesVendidas'

In [41]:
df_resultados = spark.createDataFrame(resultados)

print("=== RESULTADOS COMPLETOS ===")

df_resultados \
    .orderBy(
        "Factor",
        "Repeticion"
    ) \
    .show(
        truncate=False
    )

=== RESULTADOS COMPLETOS ===
+------+---------+-------------------+----------+--------------+----------------+----------+
|Factor|Registros|RegistrosPorSegundo|Repeticion|TiempoSegundos|UnidadesVendidas|VentaTotal|
+------+---------+-------------------+----------+--------------+----------------+----------+
|1     |15000    |27024.35           |1         |0.5551        |17142.0         |365122.0  |
|1     |15000    |24053.69           |2         |0.6236        |17142.0         |365122.0  |
|1     |15000    |36564.75           |3         |0.4102        |17142.0         |365122.0  |
|5     |75000    |194642.56          |1         |0.3853        |85710.0         |1825610.0 |
|5     |75000    |134486.85          |2         |0.5577        |85710.0         |1825610.0 |
|5     |75000    |159833.12          |3         |0.4692        |85710.0         |1825610.0 |
|10    |150000   |209649.15          |1         |0.7155        |171420.0        |3651220.0 |
|10    |150000   |230644.11          |2  

In [42]:
print("=== RESUMEN EXPERIMENTAL ===")

resumen_experimentos = (
    df_resultados
    .groupBy("Factor")
    .agg(
        F.first("Registros")
         .alias("Registros"),

        F.round(
            F.avg("TiempoSegundos"),
            4
        ).alias("TiempoPromedioSegundos"),

        F.round(
            F.min("TiempoSegundos"),
            4
        ).alias("TiempoMinimo"),

        F.round(
            F.max("TiempoSegundos"),
            4
        ).alias("TiempoMaximo"),

        F.round(
            F.stddev("TiempoSegundos"),
            4
        ).alias("DesviacionTiempo"),

        F.round(
            F.avg("RegistrosPorSegundo"),
            2
        ).alias("ThroughputPromedio"),

        F.first("UnidadesVendidas")
         .alias("UnidadesVendidas"),

        F.first("VentaTotal")
         .alias("VentaTotal")
    )
    .orderBy("Factor")
)

resumen_experimentos.show(truncate=False)

=== RESUMEN EXPERIMENTAL ===
+------+---------+----------------------+------------+------------+----------------+------------------+----------------+----------+
|Factor|Registros|TiempoPromedioSegundos|TiempoMinimo|TiempoMaximo|DesviacionTiempo|ThroughputPromedio|UnidadesVendidas|VentaTotal|
+------+---------+----------------------+------------+------------+----------------+------------------+----------------+----------+
|1     |15000    |0.5296                |0.4102      |0.6236      |0.109           |29214.26          |17142.0         |365122.0  |
|5     |75000    |0.4707                |0.3853      |0.5577      |0.0862          |162987.51         |85710.0         |1825610.0 |
|10    |150000   |0.6869                |0.6504      |0.7155      |0.0333          |218725.04         |171420.0        |3651220.0 |
+------+---------+----------------------+------------+------------+----------------+------------------+----------------+----------+



In [43]:
print("=== VALIDACIÓN DE ESCALABILIDAD ===")

for r in resultados:

    factor = r["Factor"]

    registros_esperados = 15000 * factor
    unidades_esperadas = 17142 * factor
    venta_esperada = 365122 * factor

    registros_ok = (
        r["Registros"] ==
        registros_esperados
    )

    unidades_ok = (
        abs(
            r["UnidadesVendidas"] -
            unidades_esperadas
        ) < 0.01
    )

    ventas_ok = (
        abs(
            r["VentaTotal"] -
            venta_esperada
        ) < 0.01
    )

    print(
        f"{factor}x - Repetición {r['Repeticion']} | "
        f"Registros: {registros_ok} | "
        f"Unidades: {unidades_ok} | "
        f"Ventas: {ventas_ok}"
    )

=== VALIDACIÓN DE ESCALABILIDAD ===
1x - Repetición 1 | Registros: True | Unidades: True | Ventas: True
1x - Repetición 2 | Registros: True | Unidades: True | Ventas: True
1x - Repetición 3 | Registros: True | Unidades: True | Ventas: True
5x - Repetición 1 | Registros: True | Unidades: True | Ventas: True
5x - Repetición 2 | Registros: True | Unidades: True | Ventas: True
5x - Repetición 3 | Registros: True | Unidades: True | Ventas: True
10x - Repetición 1 | Registros: True | Unidades: True | Ventas: True
10x - Repetición 2 | Registros: True | Unidades: True | Ventas: True
10x - Repetición 3 | Registros: True | Unidades: True | Ventas: True


In [44]:
print("=== COMPARACIÓN DE ESCALABILIDAD ===")

resumen_local = (
    resumen_experimentos
    .orderBy("Factor")
    .collect()
)

tiempo_base = resumen_local[0]["TiempoPromedioSegundos"]

for fila in resumen_local:

    factor = fila["Factor"]
    tiempo = fila["TiempoPromedioSegundos"]

    incremento = (
        tiempo / tiempo_base
        if tiempo_base > 0
        else 0
    )

    print(
        f"{factor}x | "
        f"Registros: {fila['Registros']} | "
        f"Tiempo promedio: {tiempo}s | "
        f"Incremento respecto a 1x: {incremento:.2f}x | "
        f"Throughput: {fila['ThroughputPromedio']} registros/s"
    )

=== COMPARACIÓN DE ESCALABILIDAD ===
1x | Registros: 15000 | Tiempo promedio: 0.5296s | Incremento respecto a 1x: 1.00x | Throughput: 29214.26 registros/s
5x | Registros: 75000 | Tiempo promedio: 0.4707s | Incremento respecto a 1x: 0.89x | Throughput: 162987.51 registros/s
10x | Registros: 150000 | Tiempo promedio: 0.6869s | Incremento respecto a 1x: 1.30x | Throughput: 218725.04 registros/s


In [45]:
import pandas as pd

resultados_pd = pd.DataFrame(resultados)

ruta_resultados = "/content/resultados_experimento_spark.csv"

resultados_pd.to_csv(
    ruta_resultados,
    index=False
)

print(
    "Archivo generado:",
    ruta_resultados
)

print(resultados_pd)

Archivo generado: /content/resultados_experimento_spark.csv
   Factor  Repeticion  Registros  TiempoSegundos  RegistrosPorSegundo  \
0       1           1      15000          0.5551             27024.35   
1       1           2      15000          0.6236             24053.69   
2       1           3      15000          0.4102             36564.75   
3       5           1      75000          0.3853            194642.56   
4       5           2      75000          0.5577            134486.85   
5       5           3      75000          0.4692            159833.12   
6      10           1     150000          0.7155            209649.15   
7      10           2     150000          0.6504            230644.11   
8      10           3     150000          0.6948            215881.85   

   UnidadesVendidas  VentaTotal  
0           17142.0    365122.0  
1           17142.0    365122.0  
2           17142.0    365122.0  
3           85710.0   1825610.0  
4           85710.0   1825610.0  
5    

In [46]:
from google.colab import files

files.download(
    "/content/resultados_experimento_spark.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [47]:
resumen_pd = (
    resumen_experimentos
    .toPandas()
)

ruta_resumen = "/content/resumen_experimento_spark.csv"

resumen_pd.to_csv(
    ruta_resumen,
    index=False
)

print(resumen_pd)

   Factor  Registros  TiempoPromedioSegundos  TiempoMinimo  TiempoMaximo  \
0       1      15000                  0.5296        0.4102        0.6236   
1       5      75000                  0.4707        0.3853        0.5577   
2      10     150000                  0.6869        0.6504        0.7155   

   DesviacionTiempo  ThroughputPromedio  UnidadesVendidas  VentaTotal  
0            0.1090            29214.26           17142.0    365122.0  
1            0.0862           162987.51           85710.0   1825610.0  
2            0.0333           218725.04          171420.0   3651220.0  


In [48]:
files.download(
    "/content/resumen_experimento_spark.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>